# Week 4: Transfer Learning, BERT (Homework)

## Question Search Engine

Embeddings are a good source of information for solving various tasks. For example, we can classify texts or find similar documents using their representations. We already know about word2vec, GloVe and fasttext, but they don't use context information from given text (only from contexts of source data).

For today we will use full power of context-aware embeddings to find text duplicates!

__Warning:__ this task assumes you have seen `seminar.ipynb`!

In [ ]:
%pip install --upgrade transformers datasets accelerate deepspeed
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
import datasets

### Data Preparation

In [ ]:
qqp = datasets.load_dataset("SetFit/qqp")
print("\n")
print("Sample[0]:", qqp["train"][0])
print("Sample[3]:", qqp["train"][3])

In [ ]:
model_name = "gchhablani/bert-base-cased-finetuned-qqp"
tokenizer = transformers.AutoTokenizer.from_pretrained(model_name)
model = transformers.AutoModelForSequenceClassification.from_pretrained(model_name)

In [ ]:
MAX_LENGTH = 128

def preprocess_function(examples):
    result = tokenizer(
        examples["text1"],
        examples["text2"],
        padding="max_length",
        max_length=MAX_LENGTH,
        truncation=True,
    )

    result["label"] = examples["label"]

    return result

In [ ]:
qqp_preprocessed = qqp.map(preprocess_function, batched=True)

In [ ]:
print(repr(qqp_preprocessed["train"][0]["input_ids"])[:100], "...")

### Evaluation (1 point)

We randomly chose a model trained on QQP - but is it any good?

One way to measure this is with validation accuracy - which is what you will implement next.

Here's the interface to help you do that:

In [ ]:
val_set = qqp_preprocessed["validation"]
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=1, shuffle=False, collate_fn=transformers.default_data_collator
)

In [ ]:
import numpy as np
from tqdm import tqdm

accuracy = 0
model = model.to('cuda')

In [ ]:
correct = 0
total = 0

for batch in tqdm(val_loader):
    with torch.no_grad():
        predicted = model(
            input_ids=batch['input_ids'].to('cuda'),
            attention_mask=batch['attention_mask'].to('cuda'),
            token_type_ids=batch['token_type_ids'].to('cuda')
        )

    preds = torch.softmax(predicted.logits, dim=1).argmax(dim=1).cpu()
    labels = batch['labels']

    correct += (preds == labels).sum().item()
    total += labels.size(0)

accuracy = correct / total
print("Accuracy:", accuracy)

**Task 1 (1 point)**

- Measure the validation accuracy of your model. Doing so naively may take several hours. Please make sure you use the following optimizations:
  - Run the model on GPU with no_grad
  - Using batch size larger than 1
  - Use optimize data loader with num_workers > 1
  - (Optional) Use [mixed precision](https://pytorch.org/docs/stable/notes/amp_examples.html)


In [ ]:
# <MY CODE HERE>
accuracy = accuracy

In [ ]:
assert 0.9 < accuracy < 0.91

### Training (4 points)

For this task, you have two options:

__Option A:__ fine-tune your own model. You are free to choose any model __except for the original BERT.__ We recommend [DeBERTa-v3](https://huggingface.co/microsoft/deberta-v3-base). Better yet, choose the best model based on public benchmarks (e.g. [GLUE](https://gluebenchmark.com/)).

You can write the training code manually or use transformers.Trainer (see [this example](https://github.com/huggingface/transformers/blob/main/examples/pytorch/text-classification)). Please make sure that your model's accuracy is at least __comparable__ with the above example for BERT.


__Option B:__ compare at least 3 pre-finetuned models (in addition to the above BERT model). For each model, report (1) its accuracy, (2) its speed, measured in samples per second in your hardware setup and (3) its size in megabytes. Please take care to compare models in equal setting, e.g. same CPU / GPU. Compile your results into a table and write a short (~half-page on top of a table) report, summarizing your findings.

**Task 2 (4 points)**
- Choose Option A or Option B (only one will be graded)
- Follow all the instructions and restrictions

In [ ]:
def evaluate(model, dataloader, device):
    model.eval()
    correct = 0
    total = 0
    start_time = time.time()
    with torch.no_grad():
        for batch in dataloader:
            batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
            inputs = {
                "input_ids": batch["input_ids"],
                "attention_mask": batch["attention_mask"],
            }
            if "token_type_ids" in batch:
                inputs["token_type_ids"] = batch["token_type_ids"]

            outputs = model(**inputs)
            preds = outputs.logits.argmax(dim=1).cpu()
            labels = batch["labels"].cpu()
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    elapsed = time.time() - start_time
    samples_per_second = total / elapsed if elapsed > 0 else float("inf")
    acc = correct / total if total > 0 else 0.0
    return {"accuracy": acc, "samples_per_sec": samples_per_second, "total_samples": total, "time_s": elapsed}

In [ ]:
# <MY CODE HERE>
import os
import time
import torch
import numpy as np
from tqdm.auto import tqdm
from torch.optim import AdamW
from torch.utils.data import DataLoader
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import default_data_collator

model_name = "microsoft/deberta-v3-small"
output_dir = "./finetuned_model"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

epochs = 2
train_batch_size = 16
eval_batch_size = 64
lr = 2e-5
weight_decay = 0.01
max_grad_norm = 1.0
save_temp_model = "/tmp/model_state.pt"

train_set = qqp_preprocessed["train"]
val_set = qqp_preprocessed["validation"]

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

train_loader = DataLoader(train_set, batch_size=train_batch_size, shuffle=True, collate_fn=default_data_collator)
val_loader = DataLoader(val_set, batch_size=eval_batch_size, shuffle=False, collate_fn=default_data_collator)

optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
total_training_steps = epochs * len(train_loader)
scheduler = transformers.get_linear_schedule_with_warmup(optimizer,
                                                        num_warmup_steps=int(0.06 * total_training_steps),
                                                        num_training_steps=total_training_steps)

scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None

best_val_acc = 0.0
for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()
    pbar = tqdm(train_loader, desc=f"Train epoch {epoch}/{epochs}", leave=False)
    for step, batch in enumerate(pbar, start=1):
        batch = {k: (v.to(device) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}

        optimizer.zero_grad()
        if scaler is not None:
            with torch.cuda.amp.autocast():
                outputs = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    token_type_ids=batch.get("token_type_ids", None),
                    labels=batch["labels"]
                )
                loss = outputs.loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                token_type_ids=batch.get("token_type_ids", None),
                labels=batch["labels"]
            )
            loss = outputs.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
            optimizer.step()

        scheduler.step()
        epoch_loss += loss.item()
        if step % 50 == 0:
            pbar.set_postfix({"loss": f"{epoch_loss/step:.4f}"})

    train_time = time.time() - t0
    avg_loss = epoch_loss / len(train_loader)
    train_samples = len(train_loader) * train_batch_size
    train_sps = train_samples / train_time if train_time > 0 else float("inf")

    val_stats = evaluate(model, val_loader, device)

    print(f"\nEpoch {epoch} summary:")
    print(f"  Train loss: {avg_loss:.4f}")
    print(f"  Train throughput: {train_sps:.1f} samples/sec ({train_samples} samples in {train_time:.1f}s)")
    print(f"  Val accuracy: {val_stats['accuracy']:.4f}")
    print(f"  Val throughput: {val_stats['samples_per_sec']:.1f} samples/sec ({val_stats['total_samples']} samples in {val_stats['time_s']:.1f}s)")

    if val_stats["accuracy"] > best_val_acc:
        best_val_acc = val_stats["accuracy"]
        os.makedirs(output_dir, exist_ok=True)
        print(f"  New best val acc {best_val_acc:.4f} — saving model to {output_dir}")
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)

torch.save(model.state_dict(), save_temp_model)
size_bytes = os.path.getsize(save_temp_model)
size_mb = size_bytes / (1024 * 1024)
print(f"\nSaved state_dict temporary size: {size_mb:.2f} MB")
print(f"Best validation accuracy during training: {best_val_acc:.4f}")
print(f"Final model folder: {output_dir}")

try:
    os.remove(save_temp_model)
except Exception:
    pass

### Finding Duplicates (1 point)

Finally, it is time to use your model to find duplicate questions.
Please implement a function that takes a question and finds top-5 potential duplicates in the training set. For now, it is fine if your function is slow, as long as it yields correct results.

Showcase how your function works with at least 5 examples.

**Task 3 (1 point)**
- Implement function for finding duplicates
- Test it on several examples (at least 5)
- Check suggested duplicates and make a conclusion about model correctness

In [ ]:
# <MY CODE HERE>
val_set = qqp_preprocessed['validation']
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=32, shuffle=False, collate_fn=transformers.default_data_collator,num_workers = 0, pin_memory = True
)

In [ ]:
from sklearn.metrics import f1_score

model.eval()

f1=0
accuracy = 0
k=5
top_k = np.array([(0,-1) for _ in range(k)]).astype(float)
from tqdm import tqdm
for batch in tqdm(val_loader):
  with torch.no_grad():
    predicted = model(
        input_ids=batch['input_ids'].to('cuda'),
        attention_mask=batch['attention_mask'].to('cuda'),
        token_type_ids=batch['token_type_ids'].to('cuda')
    )
  preds = torch.softmax(predicted.logits, dim=1).detach().cpu().data.numpy()[:,1]

  for i in range(len(preds)):
    if preds[i] > min(top_k[:,1]):
      top_k[top_k.argmin(axis=0)[1]] = (batch['idx'][i].item(),preds[i])

  f1+= f1_score(batch['labels'].numpy(),(preds >0.5).astype(int))/len(val_loader)
  accuracy += np.mean((preds >0.5).astype(int) == batch['labels'].numpy())/len(val_loader)

In [ ]:
f1,accuracy

In [ ]:
[print(i) for i in zip(val_set[top_k[:,0]]['text1'],val_set[top_k[:,0]]['text2'])];

### Bonus: Finding Duplicates Faster (0.5 point)

Try to find a way to run the function faster than just passing over all questions in a loop. For isntance, you can form a short-list of potential candidates using a cheaper method, and then run your tranformer on that short list. If you opted for this solution, please keep both the original implementation and the optimized one - and explain briefly what is the difference there.

**Bonus Task 1 (0.5 point)**
- Speed up your implementation from "Finding Duplicates" part
- Capture both old and new implementation work time
- Describe your approach

In [ ]:
<A whole lot of YOUR CODE HERE>

### Bonus: Finding Duplicates in Old-Fashioned way (1.5 points)

In this bonus task you are supposed to use pretrained embeddings (word2vec, GloVe or fasttext) for solving the duplicates problem.

**Bonus Task 2 (1.5 points)**
- Solve Finding Duplicates problem using mentioned embeddings
- Compare old-fashioned solution to previous ones (quality, speed, etc.)
- Make a small report (up to 5 steps, results and conclusions) on work done in this part

In [ ]:
<A whole lot of YOUR CODE HERE>